# Marine Science Literature: OpenAlex → Open-Access PDFs → Topic Modelling

This notebook retrieves marine-science journal articles from OpenAlex, de-duplicates and screens them, optionally parses open-access PDFs, and explores recurring themes with LDA and BERTopic. It also extracts candidate entities and relations and flags Australian marine regions. All outputs are **candidates for human review**, not findings.

**Design choices**

| Issue | How the notebook handles it |
|---|---|
| Search results span many pages | Cursor paging, up to `MAX_PER_TERM` works per query |
| Titles alone carry little information | Abstracts rebuilt from OpenAlex's inverted index; open-access PDFs downloaded and parsed with PyMuPDF |
| Boolean flags can overwrite text columns | Document-type flags are stored as `is_*` columns; text sections as `sec_*` |
| Journal names and front matter can look like papers | Screening step using OpenAlex `type`, abstract length and title rules |
| Broad search terms (for example "algae" or "eDNA") pull in non-marine work | Marine-relevance filter plus OpenAlex `type:article` and `language:en` filters |
| Choice of the number of topics must be reproducible | Selection by coherence with a stability check and a documented decision rule; held-out perplexity is reported alongside |
| Credentials | OpenAlex key and contact email are read from environment variables or Colab Secrets; no Hugging Face login is needed |

Run top to bottom. Every code cell is preceded by a plain-language explanation. Credentials must never be pasted into the notebook.


## 0. Setup
**What it does:** installs libraries and sets one configuration block. **Set `MAILTO` to your email**: OpenAlex gives faster, more reliable access to requests that identify themselves (the "polite pool"). An optional free `OPENALEX_API_KEY` can also be set if your account has one.

In [ ]:
!pip -q install requests pandas numpy matplotlib tqdm pymupdf nltk scikit-learn gensim pyLDAvis bertopic sentence-transformers umap-learn hdbscan spacy openpyxl wordcloud
!python -m spacy download en_core_web_sm -q

In [ ]:
import os, re, json, time, hashlib, warnings
from pathlib import Path
import requests, numpy as np, pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
warnings.filterwarnings("ignore")

# ---------------- CONFIG ----------------
def _secret(name):
    """Read a credential from the environment or Colab Secrets (key icon in the left sidebar). Never hard-code keys."""
    v = os.environ.get(name, "")
    if not v:
        try:
            from google.colab import userdata
            v = userdata.get(name) or ""
        except Exception:
            pass
    return v

MAILTO            = _secret("MAILTO") or "your.email@example.com"   # <-- set a real address (Colab Secret MAILTO)
OPENALEX_API_KEY  = _secret("OPENALEX_API_KEY")                    # REQUIRED since 13 Feb 2026; free at openalex.org/settings/api
if not OPENALEX_API_KEY:
    raise RuntimeError("No OpenAlex API key found. Add a Colab Secret named OPENALEX_API_KEY (enable 'Notebook access'), then re-run.")
START_YEAR, END_YEAR = 2015, 2026
MAX_PER_TERM      = 600     # works kept per search term (raise for a bigger corpus)
USE_FROZEN       = False   # True = reuse the saved raw records (outputs/raw_records_frozen.parquet) instead of calling OpenAlex again
N_SEEDS          = 10       # LDA seeds per K. The paper uses 10 (about 3.3 hours with two parallel workers); use 3 for a quick run
DOWNLOAD_PDFS     = True
MAX_PDFS          = 800     # cap for a first run; raise later
MIN_ABSTRACT_CHARS= 200

BASE = Path("/content/marine_literature") if Path("/content").exists() else Path("marine_literature")
DATA_DIR, PDF_DIR, OUT_DIR = BASE/"data", BASE/"pdfs", BASE/"outputs"
for d in (DATA_DIR, PDF_DIR, OUT_DIR): d.mkdir(parents=True, exist_ok=True)

SEARCH_TERMS = [
    "marine science", "marine ecology", "marine biology", "oceanography",
    "coral reef", "marine biodiversity", "marine conservation", "marine pollution",
    "marine ecosystem", "fisheries", "ocean warming", "ocean acidification",
    "sea turtle", "microplastics marine", "environmental DNA marine",
    "underwater acoustics", "ocean remote sensing", "seagrass", "kelp", "marine heatwave",
]

## 1. Search OpenAlex with paging and filters
**What it does:** `search_openalex` queries the OpenAlex `/works` endpoint.
- `filter=type:article` keeps journal articles (drops journal-name records, datasets, etc.).
- `language:en` keeps English text, which the NLP pipeline assumes.
- `has_abstract:true` ensures every record has text beyond the title.
- **Cursor paging** (`cursor=*`) walks through *all* results, rather than only the first page of 200.
- `select=` returns only fields we use, which makes responses much smaller and faster.

In [ ]:
OPENALEX_URL = "https://api.openalex.org/works"
SELECT = ("id,doi,title,type,publication_year,publication_date,authorships,primary_location,"
          "best_oa_location,open_access,cited_by_count,biblio,abstract_inverted_index,"
          "primary_topic,keywords,language")

def _get(params, retries=4):
    p = dict(params); p["mailto"] = MAILTO
    if OPENALEX_API_KEY: p["api_key"] = OPENALEX_API_KEY
    for k in range(retries):
        r = requests.get(OPENALEX_URL, params=p, timeout=60)
        if r.status_code == 200: return r.json()
        if r.status_code in (429, 500, 502, 503): time.sleep(2 ** k); continue
        if r.status_code == 401: raise RuntimeError('OpenAlex returned 401: API key missing, invalid, or daily usage allowance used up.')
        r.raise_for_status()
    raise RuntimeError("OpenAlex request failed repeatedly")

def search_openalex(term, start_year=START_YEAR, end_year=END_YEAR, max_results=MAX_PER_TERM, oa_only=False):
    flt = [f"from_publication_date:{start_year}-01-01", f"to_publication_date:{end_year}-12-31",
           "type:article", "language:en", "has_abstract:true"]
    if oa_only: flt.append("is_oa:true")
    cursor, out = "*", []
    while cursor and len(out) < max_results:
        data = _get({"search": term, "filter": ",".join(flt), "per-page": 200,
                     "cursor": cursor, "select": SELECT})
        out.extend(data.get("results", []))
        cursor = data.get("meta", {}).get("next_cursor")
        time.sleep(0.15)
    return out[:max_results]

## 2. Turn raw OpenAlex works into flat records
**What it does:** `reconstruct_abstract` rebuilds the abstract. OpenAlex stores abstracts as an *inverted index* (`{"word": [positions]}`), so we place each word back at its position. `extract_record` flattens one work into a row: bibliographic fields, open-access status, the best PDF link (`best_oa_location.pdf_url`), and OpenAlex's own topic classification (useful as an independent check on your models).

In [ ]:
def reconstruct_abstract(inv):
    if not inv: return ""
    pos = {}
    for word, idxs in inv.items():
        for i in idxs: pos[i] = word
    return " ".join(pos[i] for i in sorted(pos))

def extract_record(w, term):
    authors = [a["author"]["display_name"] for a in w.get("authorships", []) if a.get("author", {}).get("display_name")]
    loc  = w.get("primary_location") or {}
    src  = loc.get("source") or {}
    best = w.get("best_oa_location") or {}
    oa   = w.get("open_access") or {}
    bib  = w.get("biblio") or {}
    topic= w.get("primary_topic") or {}
    return {
        "paper_id": w.get("id"), "doi": w.get("doi"), "title": w.get("title"), "type": w.get("type"),
        "authors": authors, "author_names": "; ".join(authors),
        "publication_year": w.get("publication_year"), "publication_date": w.get("publication_date"),
        "journal": src.get("display_name"), "publisher": src.get("host_organization_name"),
        "volume": bib.get("volume"), "issue": bib.get("issue"),
        "first_page": bib.get("first_page"), "last_page": bib.get("last_page"),
        "cited_by_count": w.get("cited_by_count"),
        "is_oa": oa.get("is_oa"), "oa_status": oa.get("oa_status"),
        "landing_page_url": best.get("landing_page_url"), "pdf_url": best.get("pdf_url"),
        "license": best.get("license"),
        "abstract": reconstruct_abstract(w.get("abstract_inverted_index")),
        "openalex_topic": topic.get("display_name"),
        "openalex_field": (topic.get("field") or {}).get("display_name"),
        "search_term": term,
    }

## 3. Retrieve all search terms
**What it does:** loops over `SEARCH_TERMS`, pages through each, and builds one DataFrame. The same paper can match several terms, so we later keep a record of *every* term that found it (`matched_terms`) rather than silently dropping duplicates.

**Freezing the data:** OpenAlex is updated continuously, so repeating a search later returns slightly different records, and small changes can alter the selected number of topics. After a fresh retrieval the raw records are saved to `outputs/raw_records_frozen.parquet` (private: it contains abstracts) together with the retrieval date. Copy it to Google Drive, then set `USE_FROZEN = True` in the setup cell to repeat the analysis on exactly the same records.

In [ ]:
import datetime, json
FROZEN_RAW = OUT_DIR / "raw_records_frozen.parquet"   # PRIVATE: contains abstracts; do not publish
META_PATH  = OUT_DIR / "run_metadata.json"

if USE_FROZEN and FROZEN_RAW.exists():
    raw = pd.read_parquet(FROZEN_RAW)
    RETRIEVED_AT = json.loads(META_PATH.read_text())["retrieved_at_utc"] if META_PATH.exists() else "unknown"
    print(f"Loaded frozen raw records, originally retrieved {RETRIEVED_AT}")
else:
    RETRIEVED_AT = datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds")
    records = []
    for term in tqdm(SEARCH_TERMS, desc="OpenAlex"):
        try:
            works = search_openalex(term)
            records.extend(extract_record(w, term) for w in works)
            print(f"{term:30s} {len(works):4d} works")
        except Exception as e:
            print(f"ERROR {term}: {e}")
    raw = pd.DataFrame(records)
    raw.to_parquet(FROZEN_RAW)       # copy this file to Google Drive: Colab storage is temporary
print("Raw records:", len(raw))


## 4. De-duplicate
**What it does:** normalises DOIs and titles, then removes duplicates. It (a) keeps `matched_terms` so you can see which searches found each paper, and (b) de-duplicates by DOI first and by normalised title only for records without a DOI, avoiding accidental removal of different papers that share a generic title.

In [ ]:
def norm_doi(d):
    if not isinstance(d, str) or not d: return ""
    return re.sub(r"^https?://(dx\.)?doi\.org/", "", d.strip().lower())
def norm_title(t):
    return re.sub(r"\s+", " ", re.sub(r"[^a-z0-9\s]", " ", str(t).lower())).strip() if isinstance(t, str) else ""

raw["doi_norm"]   = raw["doi"].apply(norm_doi)
raw["title_norm"] = raw["title"].apply(norm_title)
raw["dedup_key"]  = np.where(raw["doi_norm"] != "", raw["doi_norm"], "t:" + raw["title_norm"])

terms = raw.groupby("dedup_key")["search_term"].apply(lambda s: "; ".join(sorted(set(s)))).rename("matched_terms")
corpus = raw.drop_duplicates("dedup_key").merge(terms, on="dedup_key").reset_index(drop=True)
corpus["doi_url"] = corpus["doi_norm"].apply(lambda d: f"https://doi.org/{d}" if d else None)
corpus["paper_url"] = corpus["doi_url"].fillna(corpus["landing_page_url"])
print("Unique papers:", len(corpus))

## 5. Screening
**What it does:** applies transparent, logged rules and records the *reason* for each exclusion, which you need for a PRISMA-style methods section.
1. Title looks like a journal/series name or front matter (e.g. "Annual Review of …", "Editorial").
2. Abstract shorter than `MIN_ABSTRACT_CHARS`.
3. Not marine-related: no marine keyword in title + abstract.

Document type (review / methods / model…) is assigned with `is_*` columns that **do not collide** with text columns.

In [ ]:
MARINE_RE = re.compile(r"\b(marine|ocean\w*|sea|seas|coastal|reef\w*|coral\w*|fish\w*|seagrass|kelp|mangrove\w*|"
                       r"plankton|estuar\w*|benthic|pelagic|turtle\w*|cetacean\w*|aquaculture|saltmarsh|intertidal|"
                       r"microplastic\w*|algae|algal|seaweed\w*|shelf|tidal|hydrograph\w*|underwater)\b", re.I)
FRONT_RE  = re.compile(r"^(editorial|erratum|corrigendum|retraction|preface|foreword|front matter|table of contents)\b|"
                       r"\b(journal of|annals? of|annual review of|world journal of|indian journal of)\b[^:]{0,60}$|"
                       r"^(frontiers in|marine science)$", re.I)

def screen(row):
    t, a = str(row["title"] or ""), str(row["abstract"] or "")
    if FRONT_RE.search(t.strip()):             return "exclude", "front matter / journal title"
    if len(a) < MIN_ABSTRACT_CHARS:            return "exclude", "abstract too short"
    if not MARINE_RE.search(t + " " + a):      return "exclude", "no marine terms"
    return "include", ""

corpus[["screening_decision","screening_reason"]] = corpus.apply(lambda r: pd.Series(screen(r)), axis=1)
print(corpus["screening_decision"].value_counts())
print(corpus.loc[corpus.screening_decision=="exclude","screening_reason"].value_counts())

DOC_PATTERNS = {
    "is_review":    r"\breview\b|\bsynthesis\b|\bmeta-analysis\b|\bsystematic\b|\boverview\b|\bperspective\b",
    "is_method":    r"\bmethod(s|ology)?\b|\bprotocol\b|\bframework\b|\bapproach\b",
    "is_model":     r"\bmodel(l)?ing\b|\bmodels?\b|\bsimulation\b|\bmachine learning\b|\bdeep learning\b",
    "is_mapping":   r"\bmapping\b|\bmaps?\b",
    "is_guideline": r"\bguideline(s)?\b|\brecommendations?\b",
}
txt = (corpus["title"].fillna("") + " " + corpus["abstract"].fillna("").str[:400])
for k, p in DOC_PATTERNS.items():
    corpus[k] = txt.str.contains(p, case=False, regex=True)
def doc_type(r):
    for k, lab in [("is_guideline","guidelines"),("is_review","review"),("is_model","model"),
                   ("is_method","methods"),("is_mapping","mapping")]:
        if r[k]: return lab
    return "research_article"
corpus["document_type"] = corpus.apply(doc_type, axis=1)
corpus.to_csv(DATA_DIR/"marine_master_corpus.csv", index=False)
corpus["document_type"].value_counts()

## 5b. Record the corpus snapshot
**What it does:** saves `corpus_snapshot.csv` (OpenAlex IDs, DOIs, years, matched terms and screening decisions only, with no abstracts, so it is safe to publish) and `run_metadata.json` (retrieval date, search settings, record counts and package versions). Publish these with the code so others can retrieve the same records.

In [ ]:
import importlib.metadata as md, platform
snap = corpus[["paper_id", "doi", "publication_year", "matched_terms", "screening_decision", "screening_reason"]].copy()
snap.to_csv(OUT_DIR/"corpus_snapshot.csv", index=False)

pkgs = ["requests", "pandas", "numpy", "pymupdf", "spacy", "nltk", "gensim", "scikit-learn", "pyLDAvis",
        "bertopic", "sentence-transformers", "umap-learn", "hdbscan", "matplotlib"]
versions = {}
for p in pkgs:
    try: versions[p] = md.version(p)
    except md.PackageNotFoundError: versions[p] = "not installed"
meta = {"retrieved_at_utc": RETRIEVED_AT, "python": platform.python_version(), "search_terms": SEARCH_TERMS,
        "start_year": START_YEAR, "end_year": END_YEAR, "max_per_term": MAX_PER_TERM,
        "raw_records": int(len(raw)), "unique_records": int(len(corpus)),
        "included": int((corpus.screening_decision == "include").sum()), "package_versions": versions}
META_PATH.write_text(json.dumps(meta, indent=2))
print(json.dumps({k: v for k, v in meta.items() if k != "search_terms"}, indent=2))


## 6. Download open-access PDFs (legitimately)
**What it does:** tries the PDF links OpenAlex already holds (`best_oa_location.pdf_url`), only for papers flagged open access. Each file is validated by its `%PDF` header (many "PDF" links are HTML landing pages or paywalls). Failures are logged with a reason, never retried forever.

**Sources used / available**
- **OpenAlex** `best_oa_location.pdf_url`: already in your data.
- **Unpaywall** (free, same OA data; needs your email): optional fallback below via DOI.
- **Europe PMC / PubMed Central, DOAJ, arXiv, EarthArXiv, ESSOAr, Zenodo, bioRxiv**: good extra OA sources for marine and Earth science preprints.
- Respect licences: store PDFs for text mining and cite the originals. Do not redistribute PDFs unless the licence (e.g. CC-BY) allows it.

In [ ]:
HEADERS = {"User-Agent": f"MarineLitReview/1.0 (mailto:{MAILTO})"}

def unpaywall_pdf(doi):
    if not doi: return None
    try:
        r = requests.get(f"https://api.unpaywall.org/v2/{doi}", params={"email": MAILTO}, timeout=30)
        if r.status_code == 200:
            loc = r.json().get("best_oa_location") or {}
            return loc.get("url_for_pdf")
    except Exception: pass
    return None

def download_pdf(row):
    pid = hashlib.md5(str(row["paper_id"]).encode()).hexdigest()[:16]
    path = PDF_DIR / f"{pid}.pdf"
    if path.exists() and path.stat().st_size > 5_000: return str(path), "cached"
    urls = [u for u in [row.get("pdf_url"), unpaywall_pdf(row.get("doi_norm"))] if isinstance(u, str) and u]
    for u in urls:
        try:
            r = requests.get(u, headers=HEADERS, timeout=60, stream=True, allow_redirects=True)
            if r.status_code != 200: continue
            head = next(r.iter_content(1024), b"")
            if not head.startswith(b"%PDF"): continue
            with open(path, "wb") as f:
                f.write(head)
                for chunk in r.iter_content(1 << 16): f.write(chunk)
            if path.stat().st_size > 5_000: return str(path), "ok"
            path.unlink(missing_ok=True)
        except Exception: continue
    return None, "no valid pdf"

inc = corpus[(corpus.screening_decision == "include") & (corpus.is_oa == True)]
inc = inc.sort_values("cited_by_count", ascending=False).head(MAX_PDFS)

corpus["pdf_path"], corpus["pdf_status"] = None, ""
if DOWNLOAD_PDFS:
    for idx, row in tqdm(inc.iterrows(), total=len(inc), desc="PDFs"):
        p, s = download_pdf(row)
        corpus.at[idx, "pdf_path"], corpus.at[idx, "pdf_status"] = p, s
        time.sleep(0.3)
print(corpus["pdf_status"].value_counts())

## 7. Extract text from PDFs and split into sections
**What it does:** PyMuPDF reads each PDF; we then
1. cut everything after the *References* heading (reference lists add noise and bias topics towards cited journals),
2. split the body into **Introduction / Methods / Results / Discussion** using heading patterns,
3. strip running headers, page numbers, DOIs, and URLs, which are classic "PDF-extraction artefacts" that otherwise form fake topics.

Papers without a usable PDF still enter modelling using title + abstract, so the corpus is not limited to the open-access subset.

In [ ]:
import fitz  # PyMuPDF

SEC_PATTERNS = {
    "sec_introduction": r"(?im)^\s*(\d+\.?\s*)?introduction\s*$",
    "sec_methods":      r"(?im)^\s*(\d+\.?\s*)?(materials? and methods?|methods?|methodology|study area|experimental)\s*$",
    "sec_results":      r"(?im)^\s*(\d+\.?\s*)?results?( and discussion)?\s*$",
    "sec_discussion":   r"(?im)^\s*(\d+\.?\s*)?(discussion|conclusions?)\s*$",
}
REF_RE = re.compile(r"(?im)^\s*(references?|bibliography|literature cited|acknowledg(e)?ments?)\s*$")

def pdf_to_text(path, max_pages=60):
    try:
        with fitz.open(path) as doc:
            return "\n".join(p.get_text("text") for p in doc[:max_pages])
    except Exception:
        return ""

def clean_pdf_text(t):
    t = re.sub(r"https?://\S+|www\.\S+|10\.\d{4,}/\S+", " ", t)
    t = re.sub(r"(?m)^\s*\d+\s*$", " ", t)                  # bare page numbers
    t = re.sub(r"-\n(?=[a-z])", "", t)                         # de-hyphenate line breaks
    t = re.sub(r"(?i)\b(copyright|all rights reserved|creative commons|downloaded from|licen[cs]e)\b[^\n]*", " ", t)
    return re.sub(r"\s+", " ", t)

def split_sections(raw):
    m = REF_RE.search(raw)
    body = raw[:m.start()] if m and m.start() > 0.3 * len(raw) else raw
    hits = []
    for name, pat in SEC_PATTERNS.items():
        for mm in re.finditer(pat, body): hits.append((mm.start(), mm.end(), name))
    hits.sort()
    secs = {k: "" for k in SEC_PATTERNS}
    for i, (s, e, name) in enumerate(hits):
        end = hits[i + 1][0] if i + 1 < len(hits) else len(body)
        secs[name] += " " + body[e:end]
    secs["full_text"] = body
    return {k: clean_pdf_text(v)[:60000] for k, v in secs.items()}

has_pdf = corpus["pdf_path"].notna()
sec_df = pd.DataFrame([split_sections(pdf_to_text(p)) for p in tqdm(corpus.loc[has_pdf, "pdf_path"], desc="Parse")],
                      index=corpus.index[has_pdf])
for c in sec_df.columns: corpus[c] = sec_df[c]
for c in list(SEC_PATTERNS) + ["full_text"]:
    if c not in corpus: corpus[c] = ""
    corpus[c] = corpus[c].fillna("")
corpus["has_fulltext"] = corpus["full_text"].str.len() > 2000
print("Papers with extracted full text:", int(corpus.has_fulltext.sum()), "of", len(corpus))
corpus.to_csv(DATA_DIR/"marine_corpus_with_text.csv", index=False)

## 8. Build modelling text
**What it does:** `analysis_corpus` = included papers only. `topic_text` combines **title (repeated twice to weight it) + abstract + introduction/discussion** when a PDF exists. We deliberately leave out *Methods* and *Results* for the main LDA because these sections are full of statistics and sampling vocabulary ("samples", "sites", "mean") that create generic, non-thematic topics. Methods text is kept for the method-extraction step later.

In [ ]:
analysis = corpus[corpus.screening_decision == "include"].reset_index(drop=True).copy()
analysis["topic_text"] = (analysis["title"].fillna("") + " ") * 2 + analysis["abstract"].fillna("") + " " \
                         + analysis["sec_introduction"].str[:6000] + " " + analysis["sec_discussion"].str[:6000]
print("Documents for modelling:", len(analysis))
print("With full text:", int(analysis.has_fulltext.sum()))

## 9. Clean, lemmatise, add phrases, remove stopwords
**What it does:**
- lower-cases, strips URLs/numbers/one-letter tokens;
- **lemmatises** with spaCy (`microplastics`→`microplastic`, `reefs`→`reef`) so plural/singular variants do not split into separate words;
- learns **bigrams** with Gensim `Phrases` (`coral_reef`, `sea_turtle`, `ocean_acidification`) so multi-word concepts survive;
- removes English stopwords plus **domain-generic and scholarly-boilerplate words** (study, result, figure, marine itself appears in almost every document, so it carries no discriminating information).

**Lemma check:** the filter is applied to the *lemma*, not the surface word. spaCy lemmatises "data" to "datum" and "based" to "base", so those lemmas must be listed explicitly (`datum`, `base`, `increase`, `impact`, `include`, `provide`, `na`).


In [ ]:
import re
import nltk, spacy
from nltk.corpus import stopwords
from gensim.models.phrases import Phrases, Phraser
nltk.download("stopwords", quiet=True)
nlp = spacy.load("en_core_web_sm", disable=["ner", "parser"])
nlp.max_length = 3_000_000

GENERIC = set('''study studies paper research result results method methods analysis data dataset sample samples significant
significantly observe observed use used using based show showed shown found finding findings different difference effect effects
associate associated accord number total high higher low lower fig figure table supplementary et al also may however
within among across per two one three new many many various approach review overview case potential important role
author copyright elsevier springer wiley publish publisher licence license article journal abstract introduction conclusion
marine ocean sea water cause assessment event process long-term site control find local collect information range value current
project type structure change propose support need tool individual demonstrate development region identity distribution year
occur decrease affect access improve great lead response human future reduce reveal framework understand community model
mean develop production evidence contribute diversity determine loss likely predict examine record size design report
time target condition area particularly characteristic dynamic knowledge compare distribution investigate drive challenge
relationship strategy small group suggest assess indicate integrate aalyze multiple detect long_term explore shift follow
aim depth management system consider level factor major focus apply network specific pattern key relate recent enhance highligh
evaluate effective influence evaluate effective influence evaluate effective influence field test world source value present state address'''.split())

# Lemma-level generic words. These MUST be added before STOP is built (STOP is a copy, so changing GENERIC later has no effect).
GENERIC |= set("datum base na include provide increase impact ".split())   # for cleaner topics

GENERIC |= {"datum", "base", "increase", "impact","product", "overall", "obtain", "critical", "life", "analyze", "variation",
            "discuss", "understanding", "contribution", "location", "light", "day", "form",
            "decline", "economic", "detection", "function", "well", "measure", "average",
            "rate", "policy", "know", "expriment", "work", "exist", "technique", "problem", "measurement", "long_term",
            "technique", "problem", "measurement", "highlight", "large", "growth", "limit", "set", "mechanism", "cover", "benefit", "threat",
            "context", "decade", "addition", "trend", "land", "long", "period", "conduct", "link", "establish", "advance", "effort", "performance",
            "describe", "produce", "require", "regional", "give"}

STOP = set(stopwords.words("english")) | GENERIC

def clean_lemmatise(texts, batch=64):
    out = []
    for doc in nlp.pipe((re.sub(r"[^a-zA-Z\s-]", " ", t.lower()) for t in texts), batch_size=batch):
        out.append([tok.lemma_ for tok in doc
                    if tok.is_alpha and len(tok) > 2 and tok.lemma_ not in STOP and not tok.is_stop])
    return out

tokens = clean_lemmatise(analysis["topic_text"].tolist())
bigram = Phraser(Phrases(tokens, min_count=8, threshold=12))
tokens = [bigram[t] for t in tokens]
analysis["tokens"] = tokens
print("Median tokens per doc:", int(np.median([len(t) for t in tokens])))
print("Example bigrams:", [w for w in bigram.phrasegrams][:15] if hasattr(bigram, "phrasegrams") else "n/a")

## 10. Dictionary and vocabulary quality check
**What it does:** builds the Gensim dictionary and drops words that appear in fewer than 5 documents (noise/typos) or in more than 60 % of documents (not discriminating). A frequency table and word cloud then help you spot leftover artefacts. A word cloud shows *frequency only*; themes come from the models below.

In [ ]:
from gensim import corpora
dictionary = corpora.Dictionary(tokens)
print("Vocabulary before filtering:", len(dictionary))
dictionary.filter_extremes(no_below=5, no_above=0.60)
bow = [dictionary.doc2bow(t) for t in tokens]
print("Vocabulary after filtering:", len(dictionary))

freq = pd.DataFrame({"word": [dictionary[i] for i in dictionary.cfs],
                     "frequency": list(dictionary.cfs.values())}).sort_values("frequency", ascending=False).reset_index(drop=True)
display(freq.head(40))

from wordcloud import WordCloud
wc = WordCloud(width=1600, height=900, background_color="white", max_words=150, collocations=False
              ).generate_from_frequencies(dict(zip(freq.word, freq.frequency)))
plt.figure(figsize=(14, 8)); plt.imshow(wc, interpolation="bilinear"); plt.axis("off")
plt.title("Most frequent terms in the marine science corpus"); plt.savefig(OUT_DIR/"wordcloud.png", dpi=200, bbox_inches="tight"); plt.show()

## 11. Why test 5–15 LDA topics?
Latent Dirichlet Allocation needs you to pick *K*, the number of topics.
- **Too few (K < 5)** merges unrelated fields (e.g. fisheries with climate), so topics become broad and unhelpful.
- **Too many (K > 15)** splits one theme into near-duplicates and leaves tiny topics of a handful of papers, which are hard to interpret and unstable.
- For a corpus of a few thousand abstracts, **5–15 is a standard, human-interpretable range** for a literature review (one section per theme). We scan it, then choose on evidence, not just the highest score.

**Decision rule used here (written down before looking at results):**
1. Compute C_v coherence for each K, averaged over several random seeds (`N_SEEDS`: 3 by default, 10 recommended for the final analysis, which takes about three times as long; single-seed LDA is noisy).
2. Shortlist K within 0.01–0.02 of the maximum coherence.
3. Among those, prefer the **smallest K** with no tiny (<2 %) topics and no near-duplicate topics (Jaccard of top-words < 0.3).
4. Read the topics and representative papers before accepting.

**Perplexity** is reported for every K, but it is *not* used to choose K. It is computed on a held-out 10 % of documents (a fixed split) from a twin model trained on the other 90 %, because perplexity measured on the training text always improves as K grows. Lower perplexity means the model assigns higher probability to unseen documents. It measures statistical fit, not whether topics make sense to a reader (Chang et al., 2009 found the two often disagree), so it sits alongside coherence rather than replacing it.

**Seeds:** the random seed does not control text generation. It fixes the random starting values of the training algorithm. LDA is trained by an iterative, approximate procedure, so different seeds can converge to different models; that is why each K is fitted with three seeds and the spread is reported.

**Stability warning:** coherence differs little between values of K, and it can change with the seed, the corpus snapshot and small preprocessing changes. Use more seeds, freeze the corpus (see section 3), and report the chosen K as a range if the shortlist changes between runs.

**Run time:** each (K, seed) pair needs one full model plus one coherence calculation, and the first three seeds also need a held-out twin model for perplexity. Models are fitted in parallel on the available CPU cores (`N_JOBS`), and every fitted model is cached in `outputs/lda_cache_*`, so an interrupted run resumes instead of restarting. Held-out perplexity uses the first three seeds only; coherence uses all `N_SEEDS`. With 10 seeds expect roughly 2–3 hours on a 2-core Colab runtime, and less with more cores. Copy the `outputs` folder to Google Drive before the session ends.


In [ ]:
import os, pickle, hashlib, json, time
from joblib import Parallel, delayed
from threadpoolctl import threadpool_limits
from gensim.models import LdaModel
from gensim.models.coherencemodel import CoherenceModel

K_RANGE = range(5, 16)
SEEDS = ([11, 42, 99] + list(range(100, 100 + max(N_SEEDS - 3, 0))))[:max(N_SEEDS, 1)]   # N_SEEDS is set in the setup cell
PPL_SEEDS = SEEDS[:3]                      # held-out perplexity: first three seeds only (its seed-to-seed SD is small)
N_JOBS = max(1, os.cpu_count() or 1)       # set N_JOBS = 1 if parallel runs fail or memory runs out
LDA_KW = dict(passes=15, iterations=400, alpha="auto", eta="auto", chunksize=2000)
print("Seeds:", SEEDS, "| perplexity seeds:", PPL_SEEDS, "| parallel jobs:", N_JOBS)

# Fixed 90/10 split of documents: perplexity is evaluated on documents the twin model never saw.
rng = np.random.RandomState(2026)
test_idx = set(rng.choice(len(bow), size=len(bow) // 10, replace=False).tolist())
bow_train = [b for i, b in enumerate(bow) if i not in test_idx]
bow_test  = [b for i, b in enumerate(bow) if i in test_idx]
print(f"Train docs: {len(bow_train)} | held-out docs: {len(bow_test)}")

# Every fitted model is cached on disk, so an interrupted run resumes where it stopped.
# The cache folder name changes whenever the data or LDA settings change, so stale results are never reused.
tag = hashlib.md5(json.dumps([len(bow), sorted(dictionary.token2id), LDA_KW]).encode()).hexdigest()[:8]
CACHE = OUT_DIR / f"lda_cache_{tag}"; CACHE.mkdir(exist_ok=True)

def fit_one(k, s):
    f = CACHE / f"k{k}_s{s}.pkl"
    if f.exists():
        return pickle.load(open(f, "rb"))
    with threadpool_limits(limits=1 if N_JOBS > 1 else None):
        m = LdaModel(corpus=bow, id2word=dictionary, num_topics=k, random_state=s, **LDA_KW)
        cv = CoherenceModel(model=m, texts=tokens, dictionary=dictionary, coherence="c_v", processes=1).get_coherence()
        ppl = None
        if s in PPL_SEEDS:   # held-out perplexity: gensim returns the per-word bound; perplexity = 2 ** (-bound)
            m_tr = LdaModel(corpus=bow_train, id2word=dictionary, num_topics=k, random_state=s, **LDA_KW)
            ppl = float(np.exp2(-m_tr.log_perplexity(bow_test)))
        mpath = CACHE / f"k{k}_s{s}.model"; m.save(str(mpath))
    r = {"k": k, "seed": s, "cv": float(cv), "ppl": ppl, "model_path": str(mpath)}
    pickle.dump(r, open(f, "wb"))
    return r

t0 = time.time()
tasks = [(k, s) for k in K_RANGE for s in SEEDS]
runs = Parallel(n_jobs=N_JOBS, backend="multiprocessing" if N_JOBS > 1 else "sequential", verbose=5)(
    delayed(fit_one)(k, s) for k, s in tasks)
print(f"Fitted/loaded {len(runs)} models in {(time.time() - t0) / 60:.1f} min")

results, models = [], {}
for k in K_RANGE:
    rk = [r for r in runs if r["k"] == k]
    cvs = [r["cv"] for r in rk]
    ppls = [r["ppl"] for r in rk if r["ppl"] is not None]
    m = LdaModel.load(max(rk, key=lambda r: r["cv"])["model_path"]); models[k] = m   # highest-coherence seed
    tw = [[w for w, _ in m.show_topic(t, 15)] for t in range(k)]
    div = len({w for t in tw for w in t[:10]}) / (10 * k)
    shares = np.zeros(k)
    for b in bow:
        for t, p in m.get_document_topics(b, minimum_probability=0): shares[t] += p
    shares /= shares.sum()
    jac = max(len(set(a[:10]) & set(b[:10])) / len(set(a[:10]) | set(b[:10])) for i, a in enumerate(tw) for b in tw[i+1:])
    results.append({"K": k, "cv_mean": np.mean(cvs), "cv_sd": np.std(cvs), "cv_best": max(cvs),
                    "perplexity_mean": np.mean(ppls), "perplexity_sd": np.std(ppls),
                    "diversity": div, "min_share": shares.min(), "max_share": shares.max(), "max_pair_jaccard": jac})
lda_eval = pd.DataFrame(results); display(lda_eval.round(3))
lda_eval.to_csv(OUT_DIR/"lda_evaluation.csv", index=False)


In [ ]:
import warnings; warnings.filterwarnings("ignore", category=DeprecationWarning)
fig, ax = plt.subplots(1, 3, figsize=(20, 5))
ax[0].errorbar(lda_eval.K, lda_eval.cv_mean, yerr=lda_eval.cv_sd, marker="o", capsize=3)
ax[0].set(xlabel="Number of topics (K)", ylabel=f"C_v coherence (mean ± SD, {len(SEEDS)} seeds)", title="Coherence (higher is better)"); ax[0].grid(alpha=.3)
ax[1].errorbar(lda_eval.K, lda_eval.perplexity_mean, yerr=lda_eval.perplexity_sd, marker="o", capsize=3, color="tab:green")
ax[1].set(xlabel="Number of topics (K)", ylabel=f"Held-out perplexity (mean ± SD, {len(SEEDS)} seeds)", title="Perplexity (lower is better)"); ax[1].grid(alpha=.3)
ax[2].plot(lda_eval.K, lda_eval.min_share * 100, marker="s", label="smallest topic (%)")
ax[2].plot(lda_eval.K, lda_eval.max_pair_jaccard * 100, marker="^", label="max pairwise overlap (%)")
ax[2].set(xlabel="K", title="Balance and redundancy"); ax[2].legend(); ax[2].grid(alpha=.3)
plt.tight_layout(); plt.savefig(OUT_DIR/"lda_selection.png", dpi=200); plt.show()


## 12. Compare candidate K values, then choose K
**What it does:** this cell prints the coherence of K = 7 and K = 8 (candidates examined first) next to the best K and applies the decision rule, so the choice is documented in the output rather than assumed.

In [ ]:
best_cv = lda_eval.cv_mean.max()
cand = lda_eval[(lda_eval.cv_mean >= best_cv - 0.02) & (lda_eval.min_share >= 0.02) & (lda_eval.max_pair_jaccard < 0.30)]
print("Coherence at K=7 and K=8 vs best:")
display(lda_eval[lda_eval.K.isin([7, 8])][["K","cv_mean","cv_sd","perplexity_mean","min_share","max_pair_jaccard"]].round(3))
print(f"Best mean C_v = {best_cv:.3f}")
FINAL_K = int(cand.K.min()) if len(cand) else int(lda_eval.loc[lda_eval.cv_mean.idxmax(), "K"])
print("Shortlist:", cand.K.tolist(), "→ FINAL_K =", FINAL_K)
# To override after reading the topics, set FINAL_K to another value (for example FINAL_K = 6)
final_lda = models[FINAL_K]
for t in range(FINAL_K):
    print(f"Topic {t+1}: " + ", ".join(w for w, _ in final_lda.show_topic(t, 12)))

## 13. Interactive view and representative papers (the validation step)
**What it does:** `pyLDAvis` draws topics as bubbles (distance = similarity, size = prevalence). **The numbers in front of topics** are just topic IDs, ordered by size in pyLDAvis (we set `sort_topics=False` so they match `Topic N` in the tables here, 1-based). Then we list the 5 papers with the highest probability for each topic. Read them: if the papers do not belong together, the topic is an artefact.

In [ ]:
import warnings; warnings.filterwarnings("ignore", category=DeprecationWarning)

import pyLDAvis, pyLDAvis.gensim_models as gv
vis = gv.prepare(final_lda, bow, dictionary, sort_topics=False)
pyLDAvis.save_html(vis, str(OUT_DIR/"lda_interactive.html"))
try: pyLDAvis.enable_notebook(); display(pyLDAvis.display(vis))
except Exception: pass

rows = []
for i, b in enumerate(bow):
    dist = dict(final_lda.get_document_topics(b, minimum_probability=0))
    top = max(dist, key=dist.get)
    rows.append({"idx": i, "lda_topic": top + 1, "lda_prob": dist[top], **{f"p_topic_{t+1}": dist.get(t, 0) for t in range(FINAL_K)}})
dt = pd.DataFrame(rows).set_index("idx")
analysis = analysis.drop(columns=[c for c in analysis.columns if c.startswith("p_topic_") or c in ("lda_topic","lda_prob")]).join(dt)

rep = (analysis.sort_values("lda_prob", ascending=False).groupby("lda_topic").head(5)
       .sort_values(["lda_topic","lda_prob"], ascending=[True, False]))
display(rep[["lda_topic","lda_prob","title","journal","publication_year","doi_url"]])

### Judgement checklist (answer for each topic)
1. Are the top words semantically coherent?  2. Do representative papers belong together?  3. Is it a meaningful marine-science theme?  4. Is it dominated by generic methodology words?  5. Is it a PDF-extraction artefact (publisher names, "downloaded", licence words)?  6. Too broad?  7. Duplicate of another topic?

If a topic fails (4), (5) or (7), add its offending words to `GENERIC`, re-run from Section 9, and repeat.

## 14. Topic prevalence and trends over time
**What it does:** counts dominant-topic papers, then plots each topic's **share** of that year's papers. Raw counts mostly show that more papers were indexed in later years; shares show thematic shifts. Years with very few papers (and the current, incomplete year) are flagged.

In [ ]:
import warnings; warnings.filterwarnings("ignore", category=DeprecationWarning)

In [ ]:

prev = analysis.lda_topic.value_counts().sort_index().rename_axis("topic").reset_index(name="papers")
prev["percent"] = (prev.papers / len(analysis) * 100).round(1); display(prev)
plt.figure(figsize=(9, 5)); plt.bar(prev.topic.astype(str), prev.percent)
plt.xlabel("LDA topic"); plt.ylabel("% of papers"); plt.title("Topic prevalence"); plt.savefig(OUT_DIR/"prevalence.png", dpi=200); plt.show()

ty = (analysis.groupby(["publication_year", "lda_topic"]).size().unstack(fill_value=0))
share = ty.div(ty.sum(axis=1), axis=0) * 100
ax = share.plot(figsize=(13, 6), marker="o"); ax.set(xlabel="Year", ylabel="% of that year's papers", title="Topic share over time")
ax.legend(title="Topic", bbox_to_anchor=(1.01, 1), loc="upper left"); ax.grid(alpha=.3)
plt.tight_layout(); plt.savefig(OUT_DIR/"trends.png", dpi=200); plt.show()
print("Papers per year:\n", ty.sum(axis=1).to_string())

# Simple trend test: is each topic's yearly share rising or falling?
from scipy.stats import spearmanr
trend = pd.DataFrame({t: spearmanr(share.index, share[t])[:2] for t in share.columns}, index=["rho","p"]).T.round(3)
display(trend)

## 15. BERTopic (embedding-based) on **raw** text
**What it does:** BERTopic embeds documents with a sentence transformer, reduces dimensions with UMAP, clusters with HDBSCAN, and names clusters with c-TF-IDF. Two design choices: (1) it is given **natural-language text** (title + abstract), not the stop-word-stripped tokens, because transformer embeddings need grammatical context; (2) a `CountVectorizer` removes stopwords and keeps bigrams only for *labelling*. Topic `-1` are outliers, not an error. We then **reduce** to a number comparable with LDA so the two can be compared fairly.

In [ ]:
import warnings; warnings.filterwarnings("ignore", category=DeprecationWarning)
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import CountVectorizer
from umap import UMAP
from hdbscan import HDBSCAN

docs = (analysis["title"].fillna("") + ". " + analysis["abstract"].fillna("")).tolist()
embedder = SentenceTransformer("all-MiniLM-L6-v2")
emb = embedder.encode(docs, show_progress_bar=True, batch_size=64)

topic_model = BERTopic(
    embedding_model=embedder,
    umap_model=UMAP(n_neighbors=15, n_components=5, min_dist=0.0, metric="cosine", random_state=42),
    hdbscan_model=HDBSCAN(min_cluster_size=15, min_samples=5, metric="euclidean", prediction_data=True),
    vectorizer_model=CountVectorizer(stop_words="english", ngram_range=(1, 2), min_df=5),
    calculate_probabilities=False, verbose=False)
bt_topics, _ = topic_model.fit_transform(docs, emb)
print("Raw BERTopic clusters:", len(set(bt_topics)) - (1 if -1 in bt_topics else 0),
      "| outliers:", sum(t == -1 for t in bt_topics))
topic_model.reduce_topics(docs, nr_topics=FINAL_K + 2)         # merge to a comparable number
bt_topics = topic_model.topics_
analysis["bertopic"] = bt_topics
display(topic_model.get_topic_info().head(20))

In [ ]:
topic_model.visualize_barchart(top_n_topics=FINAL_K + 1).show()
topic_model.visualize_hierarchy().show()

### 16b. Inspect BERTopic clusters with non-English text (optional check)
**What it does:** finds any BERTopic topic whose label contains Korean (Hangul) characters, exports its documents to `bertopic_odd_cluster_documents.csv`, and reports how many documents in the corpus contain Hangul and how much non-English text they hold. A topic label is built from words that occur in the documents, so a foreign-language label normally means some records contain text in that language (OpenAlex's `language:en` filter relies on its own language detection and can mislabel mixed-language records). Use the export to decide whether to keep or screen out those records.

In [ ]:
import re
hangul = re.compile(r"[\uac00-\ud7a3]")
info = topic_model.get_topic_info()
odd_topics = [int(t) for t, n in zip(info.Topic, info.Name) if hangul.search(str(n))]
print("BERTopic topics with Korean characters in their label:", odd_topics)

text_all = analysis["title"].fillna("") + " " + analysis["abstract"].fillna("")
analysis["has_hangul"] = text_all.str.contains(hangul)
analysis["nonascii_share"] = text_all.apply(lambda s: sum(ord(ch) > 127 for ch in s) / max(len(s), 1))
print("Documents in the whole corpus containing Hangul:", int(analysis.has_hangul.sum()))

sel = analysis[analysis.bertopic.isin(odd_topics)].copy()
print(f"Documents in those topics: {len(sel)} | containing Hangul: {int(sel.has_hangul.sum())}")
print("Share of non-ASCII characters in those documents:"); display(sel.nonascii_share.describe().round(3))
cols = [c for c in ["paper_id","title","publication_year","journal","doi_url","language","lda_topic","bertopic",
                    "has_hangul","nonascii_share","matched_terms"] if c in sel.columns]
sel[cols].to_csv(OUT_DIR/"bertopic_odd_cluster_documents.csv", index=False)
display(sel[cols].head(20))
if "language" in sel.columns: display(sel["language"].value_counts())


## 16. Compare LDA and BERTopic
**What it does:** a cross-tab shows which LDA themes map to which BERTopic themes. **ARI** (Adjusted Rand Index) and **NMI** (Normalised Mutual Information) quantify agreement, excluding BERTopic outliers. Rough guide: NMI ≈ 0.3–0.5 means substantial but partial overlap, which is typical because LDA assigns by word co-occurrence and BERTopic by semantic similarity. You also get an independent benchmark: agreement of each model with **OpenAlex's own field classification**.

In [ ]:
import warnings; warnings.filterwarnings("ignore", category=DeprecationWarning)
from sklearn.metrics import adjusted_rand_score as ARI, normalized_mutual_info_score as NMI
ok = analysis.bertopic != -1
print(f"LDA vs BERTopic  ARI={ARI(analysis.lda_topic[ok], analysis.bertopic[ok]):.3f}  NMI={NMI(analysis.lda_topic[ok], analysis.bertopic[ok]):.3f}  (n={ok.sum()})")
oa = analysis.openalex_topic.fillna("NA")
print(f"LDA vs OpenAlex topic  NMI={NMI(analysis.lda_topic, oa):.3f}")
print(f"BERTopic vs OpenAlex   NMI={NMI(analysis.bertopic[ok], oa[ok]):.3f}")
display(pd.crosstab(analysis.lda_topic, analysis.bertopic))

## 17. Named-entity and relation extraction (NER / RE)
**What it does:**
- A spaCy `EntityRuler` adds **marine-specific entity types** (organism, ecosystem, pollutant, method…) on top of the general model. The generic model's labels (PERSON, WORK_OF_ART and so on) are mostly meaningless for marine text.
- **Relations** are subject–verb–object triples from the dependency parse, for verbs such as *increase, reduce, affect, cause*. Treat these as **candidate** relations for human review, not facts; rule-based extraction can work for simple sentences but misses many relations, and its precision was not evaluated here.
- Input is the abstract (+ introduction when available), not the cleaned token string, because the parser needs real sentences.

In [ ]:
import warnings; warnings.filterwarnings("ignore", category=DeprecationWarning)
nlp_ner = spacy.load("en_core_web_sm")
terms = {
 "ECOSYSTEM": ["Great Barrier Reef","coral reef","seagrass meadow","mangrove forest","kelp forest","estuary","intertidal zone","deep sea","continental shelf","open ocean"],
 "ORGANISM":  ["green turtle","loggerhead turtle","sea turtle","marine turtle","coral","phytoplankton","zooplankton","seagrass","kelp","mangrove","dugong","seabird","shark","reef fish","cetacean","marine mammal"],
 "POLLUTANT": ["microplastic","microplastics","plastic pollution","marine debris","heavy metal","oil spill","nutrient pollution","ghost gear"],
 "PRESSURE":  ["ocean acidification","ocean warming","marine heatwave","coral bleaching","overfishing","sea level rise","hypoxia","eutrophication","bycatch"],
 "METHOD":    ["environmental DNA","eDNA","metabarcoding","remote sensing","acoustic telemetry","passive acoustic monitoring","stable isotope analysis","satellite imagery","machine learning","deep learning","drone","ROV"],
 "STAT_METHOD":["linear regression","generalized linear model","mixed-effects model","principal component analysis","random forest","Bayesian model","time series analysis"],
}
ruler = nlp_ner.add_pipe("entity_ruler", before="ner", config={"overwrite_ents": True})
ruler.add_patterns([{"label": l, "pattern": [{"LOWER": w.lower()} for w in t.split()]} for l, ts in terms.items() for t in ts])

TRIG = {"increase":"INCREASES","reduce":"REDUCES","decrease":"REDUCES","affect":"AFFECTS","impact":"IMPACTS","cause":"CAUSES",
        "influence":"INFLUENCES","predict":"PREDICTS","drive":"DRIVES","threaten":"THREATENS","enhance":"ENHANCES","limit":"LIMITS"}
def phrase(tok, n=8):
    span = tok.doc[tok.left_edge.i: tok.right_edge.i + 1]
    return (span if len(span) <= n else tok.doc[max(tok.i - 3, 0): tok.i + 1]).text.strip(" ,.;:")

def relations(doc, did):
    out = []
    for v in doc:
        rel = TRIG.get(v.lemma_.lower())
        if v.pos_ != "VERB" or not rel: continue
        subj = [c for c in v.children if c.dep_ in ("nsubj","nsubjpass")]
        obj  = [c for c in v.children if c.dep_ in ("dobj","obj")] + [g for p in v.children if p.dep_=="prep" for g in p.children if g.dep_=="pobj"]
        for s in subj:
            for o in obj:
                out.append({"paper_id": did, "subject": phrase(s), "relation": rel, "object": phrase(o), "sentence": v.sent.text.strip()})
    return out

ner_text = (analysis.title.fillna("") + ". " + analysis.abstract.fillna("") + " " + analysis.sec_introduction.str[:4000]).tolist()
ents, rels = [], []
for did, doc in tqdm(zip(analysis.paper_id, nlp_ner.pipe(ner_text, batch_size=16)), total=len(analysis)):
    ents += [{"paper_id": did, "entity": e.text.lower(), "type": e.label_} for e in doc.ents
             if e.label_ in terms or e.label_ in ("GPE","LOC","FAC")]
    rels += relations(doc, did)
ents, rels = pd.DataFrame(ents), pd.DataFrame(rels)
ent_freq = ents.groupby(["type","entity"]).agg(mentions=("entity","size"), papers=("paper_id","nunique")).reset_index().sort_values("papers", ascending=False)
display(ent_freq.head(40)); display(rels.relation.value_counts().to_frame("triples")); display(rels.head(15))

## 18. Geographic analysis (Australian regions)
**What it does:** flags papers mentioning Australian marine regions. It uses title + abstract + full text, because titles alone miss most mentions. Word-boundary regexes avoid false hits (for example "Victoria" in Lake Victoria, or "NSW" inside another word). Flags are *candidates* for manual checking.

In [ ]:
import warnings; warnings.filterwarnings("ignore", category=DeprecationWarning)
REGIONS = {
 "Great Barrier Reef": r"great barrier reef|\bgbr\b|coral sea",
 "Townsville / Magnetic Island": r"townsville|magnetic island|cleveland bay|palm islands",
 "Torres Strait": r"torres strait|thursday island",
 "Gulf of Carpentaria": r"gulf of carpentaria",
 "Western Australia / Ningaloo": r"ningaloo|shark bay|western australia|kimberley|rottnest",
 "New South Wales": r"new south wales|\bnsw\b|sydney|jervis bay|solitary islands",
 "Tasmania / Southern Ocean": r"tasmania|southern ocean|macquarie island",
 "South Australia": r"south australia|spencer gulf|st vincent gulf",
 "Victoria": r"port phillip|bass strait|victoria, australia|victorian coast",
 "Northern Territory": r"northern territory|darwin|arafura sea|timor sea",
 "Queensland (other)": r"moreton bay|queensland|fraser island|k'gari|whitsunday",
 "Australia-wide": r"australian waters|australian marine|australian coast|\baustralia\b",
}
full = (analysis.title.fillna("") + " " + analysis.abstract.fillna("") + " " + analysis.full_text.str[:30000]).str.lower()
for r, p in REGIONS.items(): analysis["region_" + re.sub(r"\W+", "_", r.lower()).strip("_")] = full.str.contains(p, regex=True)
reg = analysis.filter(like="region_").sum().sort_values(ascending=False).rename("papers").to_frame(); display(reg)

## 19. Literature-review table and export
**What it does:** one row per paper with bibliographic data, document type, topics, and OpenAlex's classification. Two columns (`research_objective`, `key_findings`) are left blank on purpose: fill them while you read each representative paper, since automated summaries are not reliable enough for a published review. Everything is exported as CSV and a single Excel workbook.

In [ ]:
import warnings; warnings.filterwarnings("ignore", category=DeprecationWarning)
cols = ["paper_id","title","author_names","publication_year","journal","doi_url","document_type","cited_by_count","oa_status",
        "has_fulltext","lda_topic","lda_prob","bertopic","openalex_topic","openalex_field","matched_terms"]
review = analysis[[c for c in cols if c in analysis.columns]].copy()
review["research_objective"], review["key_findings"] = "", ""
review.to_csv(OUT_DIR/"literature_review_table.csv", index=False)

top_words = pd.DataFrame([{"topic": t+1, "top_words": ", ".join(w for w,_ in final_lda.show_topic(t, 12))} for t in range(FINAL_K)])
with pd.ExcelWriter(OUT_DIR/"marine_analysis.xlsx") as xw:
    review.to_excel(xw, sheet_name="Review table", index=False)
    top_words.to_excel(xw, sheet_name="LDA topics", index=False)
    prev.to_excel(xw, sheet_name="Prevalence", index=False)
    share.round(2).to_excel(xw, sheet_name="Trends (share %)")
    lda_eval.to_excel(xw, sheet_name="LDA evaluation", index=False)
    ent_freq.to_excel(xw, sheet_name="Entities", index=False)
    rels.to_excel(xw, sheet_name="Relations", index=False)
    reg.to_excel(xw, sheet_name="Regions")
    corpus[["paper_id","doi","is_oa","pdf_url","pdf_status","screening_decision","screening_reason"]].to_excel(xw, sheet_name="Screening log", index=False)
print("Saved to", OUT_DIR)
try:
    from google.colab import files; files.download(str(OUT_DIR/"marine_analysis.xlsx"))
except Exception: pass